# 13.05 · Deploying — Milestone M5

**Chapter:** Serving & Deployment · **Milestone:** M5

## Goals
- [ ] Map the deployment options: PaaS, serverless GPU, managed model platforms, self-managed GPUs — and when each fits
- [ ] Package the app as a **container**: Dockerfile, pinned requirements, baked-in assets, non-root user, health check
- [ ] Deploy the API + UI publicly for free on **Hugging Face Spaces** (Docker Space — built in the cloud, no local Docker needed)
- [ ] Wire **CI**: tests and evals on every PR, deploy on merge
- [ ] Smoke-test a deployment and know how to roll back

## Resources
- [Hugging Face – Docker Spaces](https://huggingface.co/docs/hub/spaces-sdks-docker) · [Spaces secrets](https://huggingface.co/docs/hub/spaces-overview#managing-secrets) — free
- [Docker – Get started](https://docs.docker.com/get-started/) · [uv in Docker](https://docs.astral.sh/uv/guides/integration/docker/) — free
- [Modal docs](https://modal.com/docs) (monthly free credits; serverless GPUs) — free tier
- [GitHub Actions – Python](https://docs.github.com/en/actions/use-cases-and-examples/building-and-testing/building-and-testing-python) — free for public repos
- [The Twelve-Factor App](https://12factor.net/) — free

## Where to run it

| Option | Examples | Cost | Good for |
|---|---|---|---|
| PaaS (CPU containers) | **HF Spaces**, Render, Fly.io, Railway, Google Cloud Run | free tiers / scale-to-zero | apps that call LLM APIs (most apps!) |
| Serverless GPU | **Modal**, RunPod serverless, Replicate, Baseten, Beam | per second of GPU | your own model, bursty traffic |
| GPU rental | RunPod, Lambda, Vast.ai, Paperspace | per hour | experiments, steady self-hosting with vLLM |
| Managed model platforms | AWS Bedrock, GCP Vertex AI, Azure AI Foundry | per token / per hour | enterprise: VPC, compliance, procurement |
| Self-managed | Kubernetes (EKS/GKE/AKS) + vLLM / KServe / Ray | infra team | large scale, full control |

Rule: if your app calls an API model, deploy a **small CPU container**. Only self-host a GPU when you need your own model (12.x) or the
economics of volume make it cheaper (13.06).

In [ ]:
import atexit
import importlib.metadata as md
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import httpx

from aieng import DATA, ROOT

## 1. Twelve-factor config
Everything that differs between your laptop and production comes from **environment variables**: provider (`APP_PROVIDER`), keys
(`GEMINI_API_KEY`), allowed API keys (`APP_API_KEYS`), limits (`APP_RATE_PER_MIN`), port (`PORT`). No secrets in code or in the image.
`app.py` already reads all of these.

## 2. One process, one port
`serve.py` mounts the Gradio UI (13.04) inside the FastAPI app (13.03): API at `/v1/...`, UI at `/ui`, health at `/healthz`.
Run it like production would, then smoke-test it. The same `smoke()` runs against the deployed URL later.

In [ ]:
def smoke(base, key="dev-key"):
    """Minimal post-deploy checks: up, auth enforced, answers with sources, UI served."""
    h = httpx.get(f"{base}/healthz", timeout=30).json()
    assert h["status"] == "ok", h
    assert httpx.post(f"{base}/v1/ask", json={"question": "Who was Lincoln?"}, timeout=30).status_code == 401
    r = httpx.post(f"{base}/v1/ask", json={"question": "What is the capital of Uruguay?", "k": 3},
                   headers={"Authorization": f"Bearer {key}"}, timeout=60)
    assert r.status_code == 200 and r.json()["sources"], r.text[:300]
    assert httpx.get(f"{base}/ui/", timeout=30).status_code == 200
    print(f"✅ {base}: model {h['model']}, {r.json()['latency_ms']} ms -> {r.json()['answer'][:80]!r}")


server = subprocess.Popen([sys.executable, "-m", "uvicorn", "serve:app", "--port", "7870", "--log-level", "warning"],
                          env={**os.environ, "PORT": "7870"})
atexit.register(server.terminate)
for _ in range(120):
    try:
        httpx.get("http://localhost:7870/healthz"); break
    except httpx.ConnectError:
        time.sleep(0.5)
smoke("http://localhost:7870")
server.terminate()
_ = server.wait()

## 3. Containers in five minutes
A **container image** = your code + its exact dependencies + an OS userland, built from a `Dockerfile`, run identically anywhere.
Key ideas: layers are cached (copy `requirements.txt` and install *before* copying code, so code edits don't reinstall everything);
bake slow, static assets into the image (here: the corpus and its embeddings); run as a non-root user; one process per container.

We assemble a self-contained **build context** in `data/space/`: the three app files, the `aieng` package, pinned requirements, a Dockerfile,
and the Space's `README.md` (its YAML header tells HF to build the Dockerfile and route port 7860).

In [ ]:
SPACE = DATA / "space"
shutil.rmtree(SPACE, ignore_errors=True)
(SPACE / "aieng").mkdir(parents=True)
for f in ("app.py", "ui_gradio.py", "serve.py"):
    shutil.copy(f, SPACE / f)
for f in ("__init__.py", "llm.py", "rag.py"):
    shutil.copy(ROOT / "aieng" / f, SPACE / "aieng" / f)

# Pin exactly what works on your machine: reproducible builds, no surprise upgrades.
pkgs = ["fastapi", "uvicorn", "gradio", "httpx", "openai", "pydantic", "python-dotenv", "numpy", "datasets", "sentence-transformers", "transformers", "torch"]
(SPACE / "requirements.txt").write_text("".join(f"{p}=={md.version(p)}\n" for p in pkgs))

(SPACE / "Dockerfile").write_text("""\
FROM python:3.12-slim

# HF Spaces run containers as uid 1000
RUN useradd -m -u 1000 user
USER user
ENV HOME=/home/user PATH=/home/user/.local/bin:$PATH PYTHONUNBUFFERED=1
WORKDIR /home/user/app

# Dependencies first: this layer is cached until requirements.txt changes. CPU-only torch keeps the image ~1 GB smaller.
COPY --chown=user requirements.txt .
RUN pip install --no-cache-dir --extra-index-url https://download.pytorch.org/whl/cpu -r requirements.txt

COPY --chown=user . .
# Bake the corpus, embedding model and index into the image: fast cold starts, no downloads at runtime
RUN python -c "from aieng import rag; rag.index(); rag.embedder()"

ENV PORT=7860
EXPOSE 7860
HEALTHCHECK CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:7860/healthz')"
CMD ["uvicorn", "serve:app", "--host", "0.0.0.0", "--port", "7860"]
""")

(SPACE / "README.md").write_text("""\
---
title: mini-wiki RAG
emoji: 📚
colorFrom: blue
colorTo: green
sdk: docker
app_port: 7860
---
RAG over 3.2k Wikipedia passages: FastAPI (`/v1/ask`, `/v1/ask/stream`, docs at `/docs`) + Gradio UI at `/ui`. Built in ai-eng chapter 13.
""")
(SPACE / ".dockerignore").write_text("__pycache__/\n*.pyc\ndata/\n.env\n")

for p in sorted(SPACE.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(SPACE)!s:24} {p.stat().st_size:6} bytes")
print("\n" + (SPACE / "requirements.txt").read_text())

With Docker available locally ([OrbStack](https://orbstack.dev/) or `brew install colima docker` on a Mac) you would test the image before shipping:
```bash
cd data/space
docker build -t mini-wiki-rag .
docker run -p 7860:7860 -e GEMINI_API_KEY=$GEMINI_API_KEY -e APP_API_KEYS=dev-key mini-wiki-rag
```
Without Docker, HF builds it for you (next section) and shows the build log on the Space page.

## 4. Deploy to Hugging Face Spaces
Needs a free HF account and a **write** token (`hf auth login`, or `HF_TOKEN` in `.env`). Secrets (provider key, allowed API keys) are set on
the Space, never uploaded. The Space is **public**, so the Gradio UI is open to anyone: the per-key rate limit protects your free quota;
make it private (`private=True`) if you don't want that (then the UI needs you logged in to HF).

Flip `DEPLOY = True` to run it. A first build takes ~5–10 minutes.

In [ ]:
DEPLOY = False
SPACE_NAME = "mini-wiki-rag"

if DEPLOY:
    import secrets
    from huggingface_hub import HfApi

    api = HfApi()
    user = api.whoami()["name"]
    repo_id = f"{user}/{SPACE_NAME}"
    api.create_repo(repo_id, repo_type="space", space_sdk="docker", exist_ok=True)
    app_key = os.getenv("APP_DEPLOY_KEY") or secrets.token_urlsafe(16)
    api.add_space_secret(repo_id, "GEMINI_API_KEY", os.environ["GEMINI_API_KEY"])
    api.add_space_secret(repo_id, "APP_API_KEYS", app_key)
    api.add_space_variable(repo_id, "APP_PROVIDER", "gemini")
    api.add_space_variable(repo_id, "APP_RATE_PER_MIN", "10")
    api.upload_folder(repo_id=repo_id, repo_type="space", folder_path=SPACE, commit_message="deploy from 13.05")
    print(f"https://huggingface.co/spaces/{repo_id}  (save your API key: {app_key})")

    for _ in range(90):  # wait for the build
        stage = api.get_space_runtime(repo_id).stage
        print(stage, end=" ", flush=True)
        if stage in ("RUNNING", "BUILD_ERROR", "RUNTIME_ERROR"):
            break
        time.sleep(20)
    if stage == "RUNNING":
        smoke(f"https://{user.lower()}-{SPACE_NAME}.hf.space", key=app_key)
else:
    print("DEPLOY = False — skipping (build context is ready in data/space/)")

**Rollback** = redeploy the previous commit: every `upload_folder` is a git commit on the Space repo (see its "Files → History"), so
`api.upload_folder(..., revision=...)` or reverting in the UI restores the last good version. Same idea everywhere: immutable, versioned images.

## 5. CI/CD with GitHub Actions
In your project repo (M5 app or capstone), `.github/workflows/ci.yml`: tests + evals on every PR, deploy on merge to `main`.
Tests use a **mocked LLM** (13.03 exercise 4), so they're free and deterministic; the eval job uses real models with a small, fixed set (10.05).
```yaml
name: ci
on: {pull_request: {}, push: {branches: [main]}}
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v6
      - run: uv sync && uv run pytest -q
  eval:
    runs-on: ubuntu-latest
    needs: test
    env: {GEMINI_API_KEY: "${{ secrets.GEMINI_API_KEY }}"}
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v6
      - run: uv sync && uv run python evals/run.py --min-score 0.85   # fails the build on regression
  deploy:
    if: github.ref == 'refs/heads/main'
    needs: [test, eval]
    runs-on: ubuntu-latest
    env: {HF_TOKEN: "${{ secrets.HF_TOKEN }}"}
    steps:
      - uses: actions/checkout@v4
      - run: pip install huggingface_hub && hf upload --repo-type space you/mini-wiki-rag ./space . --commit-message "deploy ${{ github.sha }}"
      - run: python scripts/smoke.py https://you-mini-wiki-rag.hf.space
```

## 6. Other targets, briefly
**Modal** (Python-native serverless; free monthly credits; GPUs on demand). The same FastAPI app:
```python
import modal
image = modal.Image.debian_slim().pip_install_from_requirements("requirements.txt").add_local_python_source("app", "aieng")
app = modal.App("mini-wiki-rag", image=image, secrets=[modal.Secret.from_name("gemini")])

@app.function()
@modal.asgi_app()
def web():
    from serve import app as fastapi_app
    return fastapi_app
# modal deploy modal_app.py  -> https://you--mini-wiki-rag-web.modal.run
```
Your own model on a GPU with vLLM: `@app.function(gpu="L4")` + `@modal.web_server(8000)` running `vllm serve ...` — scale to zero when idle.

**Render / Fly.io / Cloud Run**: point them at the same Dockerfile; set env vars in their dashboard. **Bedrock / Vertex / Azure AI Foundry**:
the model lives in the cloud provider (call it like any API from your container) — choose them for data-residency and enterprise contracts.

## 7. Production readiness
- Health check endpoint + platform health checks; graceful shutdown
- Secrets in the platform's secret store; rotate keys; separate dev/staging/prod keys
- Tracing and logs shipped somewhere you can search (11.01); alerts on error rate, latency p95 and spend
- Rate limits and budgets per key (13.03, 11.02); guardrails on input and output (11.04)
- Versioned deploys + rollback; evals gate every deploy

## Milestone M5 — deliverable
- [ ] M3 or M4 app deployed at a **public URL**
- [ ] Eval suite running in CI, blocking regressions
- [ ] Tracing on, guardrails on, a cost/latency view (13.06)
- [ ] A README with architecture diagram, how to run locally, and eval results

## Exercises
1. Set `DEPLOY = True`, deploy, and run `smoke()` against your Space. Then break something (bad secret) and practice a rollback.
2. Deploy the same build context to Render or Google Cloud Run (both build from a Dockerfile; Cloud Run scales to zero).
3. Write the CI workflow above for your M3/M4 repo, including the mocked-LLM tests.
4. (Modal credits) Serve your fine-tuned `smollm-payments-merged` (12.06) with vLLM on a Modal GPU and call it from `aieng.llm`.

In [ ]:
# your code here